# naja-schematic: a RISC-V core

[naja-schematic](https://github.com/keplertech/naja-schematic) is an open-source schematic viewer for gate-level and RTL netlists, running in this notebook. This loads **tinyrocket**, a synthesized [Rocket](https://github.com/chipsalliance/rocket-chip) RISC-V core, with its Liberty cell library, from the [naja tutorials](https://github.com/najaeda/naja/tree/main/tutorials). Gate symbols come from the cells' Liberty functions, never from their names.

New to the viewer? Start with the [walkthrough notebook](https://colab.research.google.com/github/keplertech/naja-schematic/blob/main/python/notebooks/getting_started.ipynb).

In [ ]:
# Installs (or upgrades to) the released package. The notebook test sets
# NAJA_SCHEMATIC_NOTEBOOK_TEST to keep the freshly built wheel it runs against.
import os
if "NAJA_SCHEMATIC_NOTEBOOK_TEST" not in os.environ:
    %pip install -q -U naja-schematic

import najaeda, naja_schematic, anywidget
print("naja-schematic", naja_schematic.__version__, "| anywidget", anywidget.__version__)

## Load the design

In [ ]:
!curl -sSfL https://github.com/najaeda/naja/archive/refs/heads/main.zip -o naja-main.zip && unzip -q -o naja-main.zip "naja-main/tutorials/benchmarks/*"
# The RTL tinyrocket was synthesized from: its gates' Yosys `src` attributes
# point there, relative to this directory (OpenROAD-flow-scripts, pinned).
orfs = "https://raw.githubusercontent.com/The-OpenROAD-Project/OpenROAD-flow-scripts/43ca564bb3d4167cb62f5008c93b1ce96c7438b0/flow"
!curl -sSfL -o freechips.rocketchip.system.TinyConfig.v {orfs}/designs/src/tinyRocket/freechips.rocketchip.system.TinyConfig.v
!curl -sSfL --create-dirs -o flow/platforms/nangate45/cells_adders.v {orfs}/platforms/nangate45/cells_adders.v
bench = "naja-main/tutorials/benchmarks"
from najaeda import netlist
netlist.reset()
netlist.load_liberty([f"{bench}/liberty/tutorial_cells.lib", f"{bench}/liberty/fakeram45_64x32.lib"])
config = netlist.VerilogConfig()
config.yosys_source_attributes = True  # read the Yosys `src` attributes as source origins
top = netlist.load_verilog(f"{bench}/verilog/tinyrocket/tinyrocket.v", config=config)
top.get_name()

## Explore it
- Expand the tree on the left: instances, terms and nets load on demand.
- Right-click a flop's `D` pin → *Trace to Driver* to draw the logic cone feeding it. It stops at other flops and the top-level inputs.
- Click a hollow pin to pull in the net behind it. Scroll to zoom and drag to pan.
- Right-click a gate → *Show Sources*: its line in `tinyrocket.v`, then the Yosys origins it was synthesized from, in the RTL or the adder cells. Click one in the **Source** tab to open it at the highlighted range.

In [ ]:
view = naja_schematic.show(height=700)
view

## Back to the RTL
Which RTL made a gate? Its Yosys origins say, file and range, through najaeda. A flop, for instance:

In [ ]:
from itertools import islice
where = lambda ref: f"{ref.range.file}:{ref.range.line}:{ref.range.column}"

def first_gate_from_rtl(inst):
    for child in inst.get_child_instances():
        if not child.is_leaf():
            found = first_gate_from_rtl(child)
            if found:
                return found
        elif any(o.range.file.endswith("TinyConfig.v") for o in child.get_source_origins()):
            return child

gate = first_gate_from_rtl(top)
rtl = next(o for o in gate.get_source_origins() if o.range.file.endswith("TinyConfig.v"))
print(gate.get_name(), f"({gate.get_model_name()})")
print("  declared in the netlist at", where(gate.get_source_declaration()))
print("  made from the RTL at      ", where(rtl))
with open(rtl.range.file) as f:
    print("".join(islice(f, rtl.range.line - 1, rtl.range.line + 2)))

## Export what you've drawn as SVG
Run this after drawing something above. The SVG appears below and in `tinyrocket.svg`.

In [ ]:
from IPython.display import SVG, Markdown, display
out = display(Markdown("*Waiting for the view to draw...*"), display_id=True)
view.export_svg("tinyrocket.svg", lambda svg: out.update(SVG(svg)))